<a href="https://colab.research.google.com/github/ayush121325/Agro-weather-AI/blob/main/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/Avishkar_Hackathon/data"
OUT_DIR = "/content/drive/MyDrive/Avishkar_Hackathon/processed"
os.makedirs(OUT_DIR, exist_ok=True)

print("Files found in data folder:" , os.listdir(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files found in data folder: ['Rainfall Measurements.csv', 'soil_recommendation.csv', 'train.csv', 'test.csv']


In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
rainfall = pd.read_csv(f"{DATA_DIR}/Rainfall Measurements.csv")
soil = pd.read_csv(f"{DATA_DIR}/soil_recommendation.csv")

for name, df in [("train", train), ("test", test), ("rainfall", rainfall), ("soil", soil)]:
    print(f"\n===== {name.upper()} =====")
    print("Rows, Columns:", df.shape)
    print("Columns:", list(df.columns[:10]))
    print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])


===== TRAIN =====
Rows, Columns: (17510, 10)
Columns: ['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield']
Missing values:
 Series([], dtype: int64)

===== TEST =====
Rows, Columns: (2179, 9)
Columns: ['id', 'Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']
Missing values:
 Series([], dtype: int64)

===== RAINFALL =====
Rows, Columns: (8790, 35)
Columns: ['info', 'state', '"district"', 'month', '"1st"', '"2nd"', '"3rd"', '"4th"', '"5th"', '"6th"']
Missing values:
 Series([], dtype: int64)

===== SOIL =====
Rows, Columns: (2200, 8)
Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
Missing values:
 Series([], dtype: int64)


In [ ]:
for df in [train, test]:
    for col in ["Crop", "Season", "State"]:
        df[col] = df[col].astype(str).str.strip()


before = len(train)
train = train[(train["Area"] > 0) & (train["Production"] > 0) & (train["Yield"].notna())].copy()
print(f"Dropped {before - len(train)} invalid rows from train.")


high_yield = train[train["Yield"] > 30]["Crop"].value_counts().head(10)
print("\nTop crops with Yield > 30:\n", high_yield)


def remove_crop_typos(df):
    clean_parts = []
    for crop, group in df.groupby("Crop"):
        if len(group) >= 20:
            upper_limit = group["Yield"].quantile(0.995)
            group = group[group["Yield"] <= upper_limit]
        clean_parts.append(group)
    return pd.concat(clean_parts, ignore_index=True)

train = remove_crop_typos(train)


num_cols = ["Area", "Annual_Rainfall", "Fertilizer", "Pesticide"]
for col in num_cols:
    state_crop_med = train.groupby(["State", "Crop"])[col].transform("median")
    overall_med = train[col].median()
    train[col] = train[col].fillna(state_crop_med).fillna(overall_med)


    lookup = train.groupby(["State", "Crop"])[col].median()
    test[col] = test.apply(
        lambda r: lookup.get((r["State"], r["Crop"]), overall_med) if pd.isna(r[col]) else r[col],
        axis=1
    )

print("Clean train shape:", train.shape)
print("Clean test shape (must stay 2179):", test.shape)

Dropped 96 invalid rows from train.

Top crops with Yield > 30:
 Crop
Sugarcane    425
Coconut      148
Banana        70
Tapioca       29
Bajra         14
Onion         10
Potato         7
Ginger         5
Sesamum        2
Cashewnut      1
Name: count, dtype: int64
Clean train shape: (17305, 10)
Clean test shape (must stay 2179): (2179, 9)


In [ ]:

if "info" in rainfall.columns:
    rainfall = rainfall.drop(columns=["info"])


rainfall.columns = [c.replace('"', '').strip() for c in rainfall.columns]


for col in ["state", "district"]:
    rainfall[col] = rainfall[col].astype(str).str.replace('"', '', regex=False).str.strip()


day_cols = [c for c in rainfall.columns if c not in ["state", "district", "month"]]
rainfall["month"] = pd.to_numeric(rainfall["month"], errors="coerce")
rainfall = rainfall.dropna(subset=["month"]).copy()
rainfall["month"] = rainfall["month"].astype(int)

for col in day_cols:
    rainfall[col] = pd.to_numeric(rainfall[col], errors="coerce").fillna(0.0).clip(lower=0.0)


soil.columns = [c.strip() for c in soil.columns]
soil["label"] = soil["label"].astype(str).str.strip().str.lower()

print("Cleaned rainfall columns:", list(rainfall.columns[:8]), "... total:", len(rainfall.columns))
print("Cleaned soil crops:", soil["label"].nunique(), "unique crops")

Cleaned rainfall columns: ['state', 'district', 'month', '1st', '2nd', '3rd', '4th', '5th'] ... total: 34
Cleaned soil crops: 22 unique crops


In [ ]:
train.to_csv(f"{OUT_DIR}/clean_train.csv", index=False)
test.to_csv(f"{OUT_DIR}/clean_test.csv", index=False)
rainfall.to_csv(f"{OUT_DIR}/clean_rainfall.csv", index=False)
soil.to_csv(f"{OUT_DIR}/clean_soil.csv", index=False)

print("SUCCESS! Saved clean files to:", OUT_DIR)

SUCCESS! Saved clean files to: /content/drive/MyDrive/Avishkar_Hackathon/processed
